# 5 · Compare STAC catalog choices

**Question:** why does a reproducible catalog search need a bounded query and a provider note?

Your [STAC Browser](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/) offers WorldPop, Copernicus, Digital Earth Africa, Earth Search, and Planetary Computer presets. Browser CORS and asset authorization vary by provider. The executable examples below keep requests optional and bounded.

![STAC workflow](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-flow.svg)

## Inspect the exact presets from your client

In [ ]:
from stac_browser import PRESETS, validate_bbox
for name, url in PRESETS.items(): print(f'{name:12} {url}')

## Validate a study area

The four numbers are west, south, east, north. Invalid order and values outside WGS84 are rejected before any network call.

In [ ]:
study_bbox = validate_bbox([-123.1, 37.2, -121.8, 38.1])
print('Study bbox:', study_bbox)
for wrong in ([35,5,29,-2], [-190,0,10,5]):
    try: validate_bbox(wrong)
    except ValueError as exc: print('Rejected',wrong,'→',str(exc)[:80])

## Why page limits matter

STAC searches can return many items and assets. Start with a small page, inspect metadata, then deliberately request more. No raster pixels are downloaded by the next cell.

In [ ]:
query = {'collection':'sentinel-2-l2a','bbox':study_bbox,'start':'2024-06-01','end':'2024-06-15','limit':3}
print(json.dumps(query, indent=2) if 'json' in globals() else query)

## Optional Earth Search query

Earth Search is a public STAC API. An empty result or CORS/network error is a valid outcome to record, not a reason to silently expand the query.

In [ ]:
from stac_browser import STACBrowser
try:
    earth = await STACBrowser(PRESETS['earthsearch']).connect()
    earth_items = await earth.search(bbox=study_bbox, collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)
    print('Earth Search IDs:', [i['id'] for i in earth_items])
except Exception as exc:
    earth_items=[]
    print('Provider unavailable:', type(exc).__name__, str(exc)[:180])

## Inspect returned metadata without opening assets

In [ ]:
for item in earth_items:
    print(item['id'], '| datetime', item['properties'].get('datetime'), '| cloud', item['properties'].get('eo:cloud_cover'), '| assets', len(item.get('assets',{})))
if not earth_items: print('Use the bundled WorldPop snapshot in notebook 04 for offline analysis.')

## Optional Copernicus connection

The same collection and bbox can be tried against the Copernicus STAC API. Its dataset files may require separate authentication; a public STAC response does not grant asset access.

In [ ]:
# Run only when you want another live provider request.
# cop = await STACBrowser(PRESETS['copernicus']).connect()
# cop_items = await cop.search(bbox=study_bbox, collection='sentinel-2-l2a', start='2024-06-01', end='2024-06-15', limit=3)
# print([item['id'] for item in cop_items])
print('Copernicus query is opt-in; see the commented code above.')

## Visual comparison

Use the [STAC Browser Explorer](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/) to draw the same study area. Compare the browser's saved Search JSON with your Python `query` dictionary. The study rectangle is a query filter, not a measured data layer.

**Challenge:** change only the temporal window and predict how result count may change. Record provider, collection, bbox, dates, page limit, and retrieval time with any export.

**Citations:** [your STAC Browser](https://github.com/jltobias/JupyterLite-STAC-Browser), [Element 84 Earth Search](https://earth-search.aws.element84.com/v1), [Copernicus STAC documentation](https://documentation.dataspace.copernicus.eu/APIs/STAC.html), [STAC specification](https://stacspec.org/).